In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
EXPERIMENTS_DIR = OUTPUTS_DIR / "experiments"

RUN_ID = "latest"  # cambiar si hace falta

PRED_PATH = EXPERIMENTS_DIR / RUN_ID / "predictions.parquet"
METRICS_PATH = EXPERIMENTS_DIR / RUN_ID / "metrics.parquet"
COEFS_PATH = EXPERIMENTS_DIR / RUN_ID / "coefficients.parquet"
FEATURES_PATH = DATA_DIR / "processed" / "modeling_dataset.parquet"
FEATURE_METADATA_PATH = DATA_DIR / "processed" / "feature_metadata.csv"

ID_COL = "id_obs"
Y_TRUE_COL = "y_true"
Y_PRED_COL = "y_pred"
MODEL_COL = "model_name"
EXPERIMENT_COL = "experiment_id"
WEIGHT_COL = "sample_weight"

In [ ]:
# 1. Load backend outputs

df_models = pd.read_parquet(METRICS_PATH)
df_coefs = pd.read_parquet(COEFS_PATH)

try:
    df_feature_metadata = pd.read_csv(FEATURE_METADATA_PATH)
except FileNotFoundError:
    df_feature_metadata = pd.DataFrame({"feature_name": df_coefs["feature_name"].unique()})

required_coef_cols = ["feature_name", MODEL_COL, "coefficient"]

missing = [c for c in required_coef_cols if c not in df_coefs.columns]
if missing:
    raise ValueError(f"Missing required coefficient columns: {missing}")

df_coefs.head(), df_models.head(), df_feature_metadata.head()

FileNotFoundError: [Errno 2] No such file or directory: '/home/matias/repos/income-modeling-eph/outputs/experiments/latest/metrics.parquet'

In [ ]:
# 2. Build Ridge/Lasso analysis dataframes

OLS_MODEL = "ols"
RIDGE_MODEL = "ridge"
LASSO_MODEL = "lasso"

models_to_compare = [OLS_MODEL, RIDGE_MODEL, LASSO_MODEL]

df_coef_long = df_coefs[df_coefs[MODEL_COL].isin(models_to_compare)].copy()

if "alpha" not in df_coef_long.columns:
    df_coef_long["alpha"] = np.nan

if "feature_name" not in df_feature_metadata.columns:
    df_feature_metadata = df_feature_metadata.rename(columns={df_feature_metadata.columns[0]: "feature_name"})

df_coef_long = df_coef_long.merge(df_feature_metadata, on="feature_name", how="left")

df_coef_wide = (
    df_coef_long
    .pivot_table(
        index="feature_name",
        columns=MODEL_COL,
        values="coefficient",
        aggfunc="first"
    )
    .reset_index()
)

for m in models_to_compare:
    if m not in df_coef_wide.columns:
        df_coef_wide[m] = np.nan

df_coef_wide["abs_ols"] = df_coef_wide[OLS_MODEL].abs()
df_coef_wide["abs_ridge"] = df_coef_wide[RIDGE_MODEL].abs()
df_coef_wide["abs_lasso"] = df_coef_wide[LASSO_MODEL].abs()

df_coef_wide["ridge_shrinkage_ratio"] = (
    df_coef_wide["abs_ridge"] / df_coef_wide["abs_ols"].replace(0, np.nan)
)

df_coef_wide["lasso_shrinkage_ratio"] = (
    df_coef_wide["abs_lasso"] / df_coef_wide["abs_ols"].replace(0, np.nan)
)

df_coef_wide["lasso_zeroed"] = (
    df_coef_wide[LASSO_MODEL].fillna(0).abs() < 1e-12
)

df_alpha_path = (
    df_models[df_models[MODEL_COL].isin([RIDGE_MODEL, LASSO_MODEL])]
    .copy()
)

if "n_nonzero_features" not in df_alpha_path.columns:
    n_nonzero = (
        df_coef_long
        .assign(nonzero=lambda d: d["coefficient"].abs() > 1e-12)
        .groupby([MODEL_COL, "alpha"])["nonzero"]
        .sum()
        .reset_index(name="n_nonzero_features")
    )
    df_alpha_path = df_alpha_path.merge(
        n_nonzero,
        on=[MODEL_COL, "alpha"],
        how="left"
    )

df_coef_top = (
    df_coef_wide
    .sort_values("abs_ols", ascending=False)
    .head(50)
)

df_coef_wide.head(), df_alpha_path.head(), df_coef_top.head()

In [ ]:
DATA_STATUS = {
    "n_predictions": len(df_pred) if "df_pred" in globals() else None,
    "models": sorted(df_pred[MODEL_COL].unique().tolist()) if "df_pred" in globals() else None,
    "n_features_rows": len(df_features) if "df_features" in globals() else None,
    "feature_columns": df_features.columns.tolist() if "df_features" in globals() else None,
    "metrics_columns": df_models.columns.tolist() if "df_models" in globals() else None,
}

DATA_STATUS

In [ ]:
df_coef_long
df_coef_wide
df_alpha_path
df_coef_top